# CK-M worked solution: the empirical variant (Prairie Town's peaks)

*Energy Systems, Step by Step* · Modeler checkpoint CK-M. The design, the code, the answers to tasks 2 to 5 and 9 to 10, and the validation against EIA's published 2025 capacity factor. Put `town_year.json` (from the checkpoint page) next to this notebook first.

In [ ]:
import json, os
import numpy as np
import scipy
from scipy.optimize import linprog

print("NumPy", np.__version__, "· SciPy", scipy.__version__)
major, minor = (int(v) for v in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this notebook needs SciPy 1.9 or later (pip install -U scipy)"
from zoneinfo import ZoneInfo
import datetime as dt

## The design, before computing

- **Question:** descriptive. How much do the wind and solar farms give in the hours the town needs power most, against their year averages?
- **Metrics:** average output in each set of hours, as % of the farm's size and in MW.
- **Scaling:** demand × 450 MW, wind × 150 MW, solar × 200 MW. So 100% means SPP's highest 2025 hour, not installed capacity.
- **Hours:** the 100 highest by demand, and by net load (demand minus wind and solar); the earlier hour first where values tie.
- **Left out:** one weather year, outages, imports, an adequacy model. An average over top hours is a screen, not a capacity credit.

In [ ]:
year = json.load(open("town_year.json"))
d = np.array(year["demand"]) / 1000 * 450
w = np.array(year["wind"]) / 1000 * 150
s = np.array(year["solar"]) / 1000 * 200
net = d - w - s
# checks before any analysis
assert len(d) == len(w) == len(s) == 8760
assert d.max() == 450 and w.max() == 150 and s.max() == 200 and min(d.min(), w.min(), s.min()) >= 0
print(f"{len(d):,} hours, {d.sum() / 1000:,.1f} GWh of demand; the file's time stamps are hour-ending from {year['start_utc']} UTC")

In [ ]:
def top(x, n=100):
    return np.argsort(-x, kind="stable")[:n]        # highest first; a stable sort keeps the earlier hour first on ties

tD, tN = top(d), top(net)
cf_w, cf_s = w.mean() / 150 * 100, s.mean() / 200 * 100
rows = {"whole year": (cf_w, cf_s, w.mean() + s.mean()),
        "100 highest demand": (w[tD].mean() / 1.5, s[tD].mean() / 2, (w[tD] + s[tD]).mean()),
        "100 highest net load": (w[tN].mean() / 1.5, s[tN].mean() / 2, (w[tN] + s[tN]).mean())}
for k, (a, b, c) in rows.items():
    print(f"{k:22s} wind {a:5.2f}%  solar {b:5.2f}%  both {c:6.2f} MW")
assert abs(cf_w - 52.75) < 0.01 and abs(rows["100 highest demand"][0] - 58.11) < 0.01 and abs(rows["100 highest demand"][1] - 68.97) < 0.01
assert abs(rows["100 highest net load"][2] - 52.89) < 0.01

**Where the hours fall.** The file's index is the hour ending at `start_utc` plus *i* hours, in UTC; convert to Central time to read the hours.

In [ ]:
t0 = dt.datetime.fromisoformat(year["start_utc"]).replace(tzinfo=dt.timezone.utc)
begin = lambda i: (t0 + dt.timedelta(hours=int(i) - 1)).astimezone(ZoneInfo("America/Chicago"))   # hour beginning, local
for name, idx in [("demand", tD), ("net load", tN)]:
    months = sorted({begin(i).month for i in idx}); hours = sorted({begin(i).hour for i in idx})
    print(f"100 highest {name}: months {months}, local hours beginning {hours}")

## Validation against a published benchmark (tasks 9 to 11)

EIA's *Electric Power Monthly*, Table 6.07.B, gives 33.8% for the U.S. utility-scale wind fleet in 2025 and 24.0% for solar. The dataset's wind figure is far above it: its shapes are shares of their own highest hour, not of capacity, and SPP's fleet and wind differ from the U.S. average. These data alone cannot split the gap. What survives a linear rescaling to 33.8% is every ratio to the farm's own average.

In [ ]:
gap = cf_w - 33.8
k = 0.338 / (cf_w / 100)
w2 = w * k
ratio, ratio2 = w[tN].mean() / w.mean(), w2[tN].mean() / w2.mean()
print(f"Wind: dataset {cf_w:.2f}% against EIA 33.8%: {gap:.2f} percentage points; solar {cf_s:.2f}% against 24.0%")
print(f"Net-load hours ÷ year average: {ratio:.4f} before rescaling, {ratio2:.4f} after")
assert abs(gap - 18.95) < 0.01 and abs(ratio - 0.5606) < 1e-4 and abs(ratio - ratio2) < 1e-12